# 运行代理
代理支持同步和异步执行，使用 `.invoke()` / `await .ainvoke()` 获得完整响应，或使用 `.stream()` / `.astream()` 获得增量流输出。本节将解释如何提供输入、解释输出、启用流式传输以及控制执行限制。

## 基本用法¶
代理可以通过两种主要模式执行：

- 同步使用`.invoke()`或`.stream()`
- 使用 `await .ainvoke()` 进行异步操作，或使用 `.astream()` 进行异步操作



In [ ]:
from langgraph.prebuilt import create_react_agent

agent = create_react_agent(...)

response = agent.invoke({"messages": [{"role": "user", "content": "what is the weather in sf"}]})

## 输入和输出

代理使用的语言模型需要将信息列表作为输入。因此，代理输入和输出存储为代理状态下的`messages`键下的`messages`列表。


代理输入必须是带有`messages`键的字典。支持的格式包括：


| 格式 | 例子 |
| :--- | :--- |
| 细绳 | ｛＂messages＂：＂Hello＂} 一解释为HumanMessage |
| 消息字典 | ｛＂messages＂：｛＂role＂：＂user＂，＂content＂：＂Hello＂}} |
| 消息列表 | ｛＂messages＂：［｛＂role＂：＂user＂，＂content＂：＂Hello＂}]} |
| 具有自定义状态 | ｛＂messages＂：［｛＂role＂：＂user＂，＂content＂：＂Hello＂}], "user_name": "Alice"} — 如果使用自定义 state＿schema |

消息会自动转换为 LangChain 的内部消息格式。您可以在 LangChain 文档中了解更多关于LangChain 消息的信息。


> 您可以直接在输入字典中提供代理状态架构中定义的其他字段。这允许基于运行时数据或先前工具输出的动态行为。有关完整详情，
请参阅上下文指南。

> 输入信息的字符串会被转换为 `HumanMessage`。这种行为与 `create_react_agent` 中的 `prompt` 参数不同，后者在以字符串形式传递时被解释为 `SystemMessage`。

## 输出格式¶
代理输出是一个包含以下内容的字典：

- messages：执行期间交换的所有消息的列表（用户输入、助手回复、工具调用）。
- 可选地，如果structured_response配置了[结构化输出](https://langchain-ai.github.io/langgraph/agents/agents/#6-configure-structured-output)。
- 如果使用自定义state_schema，则输出中可能还会显示与您定义的字段对应的附加键。这些键可以保存工具执行或提示逻辑中更新的状态值。

有关使用自定义状态模式和访问上下文的更多详细信息，请参阅[上下文指南](https://langchain-ai.github.io/langgraph/agents/context/)。


## 流式输出¶
代理支持流式响应，以实现更灵敏的应用程序。其中包括：

- 每一步后更新进度
- 生成的LLM 令牌
- 执行期间的自定义工具消息

流式传输可在同步和异步模式下使用：

In [ ]:
for chunk in agent.stream(
    {"messages": [{"role": "user", "content": "what is the weather in sf"}]},
    stream_mode="updates"
):
    print(chunk)


async for chunk in agent.astream(
    {"messages": [{"role": "user", "content": "what is the weather in sf"}]},
    stream_mode="updates"
):
    print(chunk)

> 有关完整详细信息，请参阅[流媒体指南](https://langchain-ai.github.io/langgraph/how-tos/streaming/)。

## 最大迭代次数；
为控制代理执行并避免无限循环，可设置递归限制。这定义了在引发 `GraphRecursionError` 之前代理可执行的最大步数。可以在运行时或通过 `.with_config()` 定义代理时配置递归限制：



In [ ]:
### Runtime
from langgraph.errors import GraphRecursionError
from langgraph.prebuilt import create_react_agent

def get_weather(location: str) -> str:
    """A tool to get the weather for a given location."""
    return f"The weather in {location} is sunny."

max_iterations = 3
recursion_limit = 2 * max_iterations + 1
agent = create_react_agent(
    model="anthropic:claude-3-5-haiku-latest",
    tools=[get_weather]
)

try:
    response = agent.invoke(
        {"messages": [{"role": "user", "content": "what's the weather in sf"}]},
        {"recursion_limit": recursion_limit},
    )
except GraphRecursionError:
    print("Agent stopped due to max iterations.")

In [ ]:
### .with_config()

from langgraph.errors import GraphRecursionError
from langgraph.prebuilt import create_react_agent

max_iterations = 3
recursion_limit = 2 * max_iterations + 1
agent = create_react_agent(
    model="anthropic:claude-3-5-haiku-latest",
    tools=[get_weather]
)
agent_with_recursion_limit = agent.with_config(recursion_limit=recursion_limit)

try:
    response = agent_with_recursion_limit.invoke(
        {"messages": [{"role": "user", "content": "what's the weather in sf"}]},
    )
except GraphRecursionError:
    print("Agent stopped due to max iterations.")


## 其他资源¶ 
- LangChain 中的[异步编程](https://python.langchain.com/docs/concepts/async?_gl=1*14smfjy*_gcl_au*NDE1NjY4Mjc0LjE3NTM0Mjc3MTc.*_ga*MTEyMjY1OTA5MS4xNzUzNDI3NzE4*_ga_47WX3HKKY2*czE3NTM1ODE1MzEkbzIkZzAkdDE3NTM1ODE1MzEkajYwJGwwJGgw)